In [1]:
import os
os.getcwd()

'd:\\Aamir Gulzar\\KSA_project2\\Cancer-detection-classifier\\slide_aggregation\\caption_generation'

In [2]:
import pandas as pd

csv_file = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_aggregation\caption_generation\Results\classification_results_tcga_15_classes.csv"
df = pd.read_csv(csv_file)
df.head()
# Assuming 'image_id' is the column with image identifiers and 'label' is the column with labels

,patch_name,label,score
0,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159.png,ADI,0.527358
1,TCGA-3L-AA1B_nonMSIH_x10288_y13024_patch00160.png,ADI,0.457169
2,TCGA-3L-AA1B_nonMSIH_x10288_y13536_patch00161.png,LYA,0.575389
3,TCGA-3L-AA1B_nonMSIH_x10288_y14048_patch00162.png,LYA,0.567092
4,TCGA-3L-AA1B_nonMSIH_x10288_y14560_patch00163.png,MUS,0.332682


## TCGA Data Aggregation

In [3]:
import pandas as pd
import torch
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict
from tqdm import tqdm

# ─── CONFIG ───────────────────────────────────────────────────────────────────
csv_file     = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_aggregation\caption_generation\Results\classification_results_tcga.csv"
features_dir = r"D:\Aamir Gulzar\KSA_project2\dataset\Features\Conch_v1_fivecrop"
output_dir   = r"D:\Aamir Gulzar\KSA_project2\dataset\slide_aggregation\Caption_based_aggregation\ConchV1"

FEATURE_DIM    = 512
EXPECTED_SHAPE = torch.Size([5, 512])
MAX_WORKERS    = 12
# ──────────────────────────────────────────────────────────────────────────────

all_groups = ['ADI', 'DEB', 'LYM', 'PLC', 'LYA', 'MUC', 'MUS', 'NORM',
              'ADE', 'STR', 'MES', 'CAR', 'PDC', 'SIG']

os.makedirs(output_dir, exist_ok=True)

# ── Step 1: Load CSV + build all paths VECTORIZED ─────────────────────────────
print("📂 Loading CSV...")
df = pd.read_csv(csv_file)
print(f"  Total rows     : {len(df):,}")

df['patch_name_no_ext'] = df['patch_name'].str.replace('.png', '', regex=False)
df['slide_name']        = df['patch_name'].str.split('_').str[:2].str.join('_')
df['feature_path']      = (features_dir + os.sep
                           + df['slide_name'] + os.sep
                           + df['patch_name_no_ext'] + '.pt')

print(f"  Unique slides  : {df['slide_name'].nunique():,}")
print(f"  Labels found   : {sorted(df['label'].unique())}\n")

# ── Step 2: Filter missing files ──────────────────────────────────────────────
print("🔍 Filtering missing files...")
df['exists'] = df['feature_path'].apply(os.path.exists)
print(f"  ✓ Found   : {df['exists'].sum():,}")
print(f"  ✗ Missing : {(~df['exists']).sum():,}\n")
df = df[df['exists']].reset_index(drop=True)

# ── Step 3: Parallel file loader ──────────────────────────────────────────────
def load_one(args):
    feature_path, slide_name, label = args
    try:
        tensor = torch.load(feature_path, map_location="cpu", weights_only=True)
        if tensor.shape != EXPECTED_SHAPE:
            return None
        avg = tensor.mean(dim=0)   # (512,)
        return (slide_name, label, avg)
    except Exception:
        return None

print(f"⚡ Loading {len(df):,} feature files using {MAX_WORKERS} threads...")

args_list = list(zip(df['feature_path'], df['slide_name'], df['label']))

slide_features = defaultdict(lambda: defaultdict(list))

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(load_one, args): args for args in args_list}
    for future in tqdm(as_completed(futures), total=len(args_list), desc="Loading"):
        result = future.result()
        if result is not None:
            slide_name, label, avg = result
            slide_features[slide_name][label].append(avg)

print(f"\n  ✓ Loaded features for {len(slide_features):,} slides\n")

# ── Step 4: Aggregate and save ────────────────────────────────────────────────
print("🔗 Aggregating and saving slide-level features...")

for slide_name, group_data in tqdm(slide_features.items(), desc="Aggregating"):
    final_features = []

    for group in all_groups:
        if group_data[group]:
            stacked      = torch.stack(group_data[group])         # (N, 512)
            avg_features = stacked.mean(dim=0, keepdim=True)      # (1, 512)
        else:
            avg_features = torch.zeros(1, FEATURE_DIM)            # (1, 512)

        final_features.append(avg_features)

    final_tensor = torch.cat(final_features, dim=0)               # (14, 512)

    output_file = os.path.join(output_dir, f"{slide_name}.pt")
    torch.save(final_tensor, output_file)

print(f"\n✅ Done. All slide features saved to:\n  {output_dir}")
# ```

# ---

# ## Why It Was Stuck for 21 Minutes
# ```
# iterrows() on 1.3M rows
#     │
#     ├── pandas overhead per row       → ~1.3M object creations
#     ├── os.path.exists per row        → ~1.3M filesystem calls, sequential
#     └── torch.load per row            → ~1.3M file open/close, sequential

# All of this blocking on a single thread = 21 minutes just to START seeing output
# ```

# ## What's Different Now
# ```
# Before                          After
# ──────────────────────────────────────────────────────
# iterrows() 1.3M rows            vectorized str ops
# os.path.exists in loop          vectorized apply upfront
# torch.load sequential           12 parallel threads
# no progress bar                 tqdm progress on both steps

📂 Loading CSV...
  Total rows     : 1,377,933
  Unique slides  : 417
  Labels found   : ['ADE', 'ADI', 'CAR', 'DEB', 'LYA', 'LYM', 'MES', 'MUC', 'MUS', 'NORM', 'PDC', 'PLC', 'SIG', 'STR']

🔍 Filtering missing files...
  ✓ Found   : 1,377,933
  ✗ Missing : 0

⚡ Loading 1,377,933 feature files using 12 threads...


Loading: 100%|██████████| 1377933/1377933 [08:19<00:00, 2760.10it/s]



  ✓ Loaded features for 417 slides

🔗 Aggregating and saving slide-level features...


Aggregating: 100%|██████████| 417/417 [00:01<00:00, 325.11it/s]


✅ Done. All slide features saved to:
  D:\Aamir Gulzar\KSA_project2\dataset\slide_aggregation\Caption_based_aggregation\ConchV1


## TCGA data aggregation updated for ease of use with all models

In [3]:
import pandas as pd
import torch
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict
from tqdm import tqdm
tqdm.pandas()

# ─── CONFIG ───────────────────────────────────────────────────────────────────
ACTIVE_MODELS = [
    "ConchV1",
    "Conch1_5",
    "H-Optimus-1",
    "UNI2",
    "Virchow2"
]

CSV_FILE = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_aggregation\caption_generation\Results\classification_results_tcga_15_classes.csv"

MODEL_CONFIG = {
    "ConchV1": {
        "features_dir"  : r"D:\Aamir Gulzar\KSA_project2\dataset\Features\Conch_v1_fivecrop",
        "output_dir"    : r"D:\Aamir Gulzar\KSA_project2\dataset\slide_aggregation\Caption_based_aggregation_15_classes\ConchV1",
        "feature_dim"   : 512,
        "expected_shape": torch.Size([5, 512]),
    },
    "Conch1_5": {
        "features_dir"  : r"D:\Aamir Gulzar\KSA_project2\dataset\Features\Conch_v1_5_fivecrop_Updated",
        "output_dir"    : r"D:\Aamir Gulzar\KSA_project2\dataset\slide_aggregation\Caption_based_aggregation_15_classes\Conch1_5",
        "feature_dim"   : 768,
        "expected_shape": torch.Size([5, 768]),
    },
    "H-Optimus-1": {
        "features_dir"  : r"D:\Aamir Gulzar\KSA_project2\dataset\Features\H_Optimus_1_fivecrop_Updated",
        "output_dir"    : r"D:\Aamir Gulzar\KSA_project2\dataset\slide_aggregation\Caption_based_aggregation_15_classes\H-Optimus-1",
        "feature_dim"   : 1536,
        "expected_shape": torch.Size([5, 1536]),
    },
    "UNI2": {
        "features_dir"  : r"D:\Aamir Gulzar\KSA_project2\dataset\Features\Uni_2h_fivecrop_Updated",
        "output_dir"    : r"D:\Aamir Gulzar\KSA_project2\dataset\slide_aggregation\Caption_based_aggregation_15_classes\UNI2",
        "feature_dim"   : 1536,
        "expected_shape": torch.Size([5, 1536]),
    },
    "Virchow2": {
        "features_dir"  : r"D:\Aamir Gulzar\KSA_project2\dataset\Features\Virchow2_fivecrop",
        "output_dir"    : r"D:\Aamir Gulzar\KSA_project2\dataset\slide_aggregation\Caption_based_aggregation_15_classes\Virchow2",
        "feature_dim"   : 2560,
        "expected_shape": torch.Size([5, 2560]),
    },
}

ALL_GROUPS = [
    'ADI', 'DEB', 'TIL', 'PLC', 'LYA', 'LYM', 'MUS', 'MUC',
    'NORM', 'ADE', 'STR', 'CAR', 'PDC', 'SIG', 'BACK'
]

MAX_WORKERS = 12
# ──────────────────────────────────────────────────────────────────────────────


def run_for_model(model_name: str, cfg: dict, df_base: pd.DataFrame):
    print(f"\n{'═'*60}")
    print(f"  MODEL: {model_name}")
    print(f"{'═'*60}")

    features_dir   = cfg["features_dir"]
    output_dir     = cfg["output_dir"]
    feature_dim    = cfg["feature_dim"]
    expected_shape = cfg["expected_shape"]

    os.makedirs(output_dir, exist_ok=True)

    # ── Build paths ────────────────────────────────────────────────────────────
    df = df_base.copy()
    df['feature_path'] = (features_dir + os.sep
                          + df['slide_name'] + os.sep
                          + df['patch_name_no_ext'] + '.pt')

    # ── Filter missing ─────────────────────────────────────────────────────────
    df['exists'] = df['feature_path'].progress_apply(os.path.exists)
    print(f"  Found : {df['exists'].sum():,}  |  Missing: {(~df['exists']).sum():,}")
    df = df[df['exists']].reset_index(drop=True)

    # ── Parallel load ──────────────────────────────────────────────────────────
    def load_one(args):
        feature_path, slide_name, label = args
        try:
            tensor = torch.load(feature_path, map_location="cpu", weights_only=True)
            if tensor.shape != expected_shape:
                return None
            return (slide_name, label, tensor.mean(dim=0))
        except Exception:
            return None

    args_list = list(zip(df['feature_path'], df['slide_name'], df['label']))
    slide_features = defaultdict(lambda: defaultdict(list))

    print(f"  Loading {len(df):,} files with {MAX_WORKERS} threads...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(load_one, a): a for a in args_list}
        for future in tqdm(as_completed(futures), total=len(args_list), desc=f"  {model_name}"):
            result = future.result()
            if result is not None:
                slide_name, label, avg = result
                slide_features[slide_name][label].append(avg)

    print(f"  Loaded features for {len(slide_features):,} slides")

    # ── Aggregate + save ───────────────────────────────────────────────────────
    n_groups = len(ALL_GROUPS)
    for slide_name, group_data in tqdm(slide_features.items(), desc=f"  Aggregating {model_name}"):
        final_features = []
        for group in ALL_GROUPS:
            if group_data[group]:
                avg_features = torch.stack(group_data[group]).mean(dim=0, keepdim=True)
            else:
                avg_features = torch.zeros(1, feature_dim)
            final_features.append(avg_features)

        final_tensor = torch.cat(final_features, dim=0)   # (n_groups, feature_dim)
        torch.save(final_tensor, os.path.join(output_dir, f"{slide_name}.pt"))

    print(f"  ✓ Saved {len(slide_features):,} slide tensors ({n_groups} × {feature_dim}) → {output_dir}")


def main():
    print("📂 Loading CSV...")
    df = pd.read_csv(CSV_FILE)
    print(f"  Total rows     : {len(df):,}")

    df['patch_name_no_ext'] = df['patch_name'].str.replace('.png', '', regex=False)
    df['slide_name']        = df['patch_name'].str.split('_').str[:2].str.join('_')

    print(f"  Unique slides  : {df['slide_name'].nunique():,}")
    print(f"  Labels found   : {sorted(df['label'].unique())}")
    print(f"\n  Active models  : {ACTIVE_MODELS}")

    for model_name in ACTIVE_MODELS:
        if model_name not in MODEL_CONFIG:
            print(f"\n[WARN] '{model_name}' not in MODEL_CONFIG — skipping.")
            continue
        run_for_model(model_name, MODEL_CONFIG[model_name], df)

    print(f"\n{'═'*60}")
    print("✅ ALL DONE")
    print(f"{'═'*60}")


if __name__ == "__main__":
    main()

📂 Loading CSV...
  Total rows     : 1,355,579
  Unique slides  : 417
  Labels found   : ['ADE', 'ADI', 'BACK', 'CAR', 'DEB', 'LYA', 'LYM', 'MUC', 'MUS', 'NORM', 'PDC', 'PLC', 'SIG', 'STR', 'TIL']

  Active models  : ['ConchV1', 'Conch1_5', 'H-Optimus-1', 'UNI2', 'Virchow2']

════════════════════════════════════════════════════════════
  MODEL: ConchV1
════════════════════════════════════════════════════════════
  Found : 1,351,184  |  Missing: 4,395
  Loading 1,351,184 files with 12 threads...


  ConchV1: 100%|██████████| 1351184/1351184 [24:28<00:00, 920.29it/s] 


  Loaded features for 417 slides


  Aggregating ConchV1: 100%|██████████| 417/417 [00:02<00:00, 197.00it/s]


  ✓ Saved 417 slide tensors (15 × 512) → D:\Aamir Gulzar\KSA_project2\dataset\slide_aggregation\Caption_based_aggregation_15_classes\ConchV1

════════════════════════════════════════════════════════════
  MODEL: Conch1_5
════════════════════════════════════════════════════════════
  Found : 1,351,184  |  Missing: 4,395
  Loading 1,351,184 files with 12 threads...


...

  ✓ Saved 417 slide tensors (15 × 1536) → D:\Aamir Gulzar\KSA_project2\dataset\slide_aggregation\Caption_based_aggregation_15_classes\UNI2

════════════════════════════════════════════════════════════
  MODEL: Virchow2
════════════════════════════════════════════════════════════
  Found : 1,351,184  |  Missing: 4,395
  Loading 1,351,184 files with 12 threads...


  Virchow2: 100%|██████████| 1351184/1351184 [19:03<00:00, 1182.02it/s]


  Loaded features for 417 slides


  Aggregating Virchow2: 100%|██████████| 417/417 [00:04<00:00, 95.22it/s] 


  ✓ Saved 417 slide tensors (15 × 2560) → D:\Aamir Gulzar\KSA_project2\dataset\slide_aggregation\Caption_based_aggregation_15_classes\Virchow2

════════════════════════════════════════════════════════════
✅ ALL DONE
════════════════════════════════════════════════════════════


## Aggregation for PAIP data for all models

In [4]:
import pandas as pd
import torch
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict
from tqdm import tqdm

# ─── CONFIG ───────────────────────────────────────────────────────────────────
ACTIVE_MODELS = [
    "ConchV1",
    "Conch1_5",
    "H-Optimus-1",
    "UNI2",
    "Virchow2"
]

CSV_FILE = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_aggregation\caption_generation\Results\classification_results_paip_15_classes.csv"

MODEL_CONFIG = {
    "ConchV1": {
        "features_dir"  : r"D:\Aamir Gulzar\KSA_project2\paip_data\Features\Conch_v1_fivecrop_Updated",
        "output_dir"    : r"D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_based_aggregation_15_classes\ConchV1",
        "feature_dim"   : 512,
        "expected_shape": torch.Size([5, 512]),
    },
    "Conch1_5": {
        "features_dir"  : r"D:\Aamir Gulzar\KSA_project2\paip_data\Features\Conch_v1_5_fivecrop_Updated",
        "output_dir"    : r"D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_based_aggregation_15_classes\Conch1_5",
        "feature_dim"   : 768,
        "expected_shape": torch.Size([5, 768]),
    },
    "H-Optimus-1": {
        "features_dir"  : r"D:\Aamir Gulzar\KSA_project2\paip_data\Features\H_Optimus_1_fivecrop_Updated",
        "output_dir"    : r"D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_based_aggregation_15_classes\H-Optimus-1",
        "feature_dim"   : 1536,
        "expected_shape": torch.Size([5, 1536]),
    },
    "UNI2": {
        "features_dir"  : r"D:\Aamir Gulzar\KSA_project2\paip_data\Features\Uni_2h_fivecrop_Updated",
        "output_dir"    : r"D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_based_aggregation_15_classes\UNI2",
        "feature_dim"   : 1536,
        "expected_shape": torch.Size([5, 1536]),
    },
    "Virchow2": {
        "features_dir"  : r"D:\Aamir Gulzar\KSA_project2\paip_data\Features\Virchow2_fivecrop_Updated",
        "output_dir"    : r"D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_based_aggregation_15_classes\Virchow2",
        "feature_dim"   : 2560,
        "expected_shape": torch.Size([5, 2560]),
    },
}

ALL_GROUPS = [
    'ADI', 'DEB', 'TIL', 'PLC', 'LYA', 'LYM', 'MUS', 'MUC',
    'NORM', 'ADE', 'STR', 'CAR', 'PDC', 'SIG', 'BACK'
]

MAX_WORKERS = 12
# ──────────────────────────────────────────────────────────────────────────────


def run_for_model(model_name: str, cfg: dict, df_base: pd.DataFrame):
    print(f"\n{'═'*60}")
    print(f"  MODEL: {model_name}")
    print(f"{'═'*60}")

    features_dir   = cfg["features_dir"]
    output_dir     = cfg["output_dir"]
    feature_dim    = cfg["feature_dim"]
    expected_shape = cfg["expected_shape"]

    os.makedirs(output_dir, exist_ok=True)

    # ── Build paths (4-part slide name for PAIP) ───────────────────────────────
    df = df_base.copy()
    df['feature_path'] = (features_dir + os.sep
                          + df['slide_name'] + os.sep
                          + df['patch_name_no_ext'] + '.pt')

    # ── Filter missing ─────────────────────────────────────────────────────────
    df['exists'] = df['feature_path'].apply(os.path.exists)
    print(f"  Found : {df['exists'].sum():,}  |  Missing: {(~df['exists']).sum():,}")
    df = df[df['exists']].reset_index(drop=True)

    # ── Parallel load ──────────────────────────────────────────────────────────
    def load_one(args):
        feature_path, slide_name, label = args
        try:
            tensor = torch.load(feature_path, map_location="cpu", weights_only=True)
            if tensor.shape != expected_shape:
                return None
            return (slide_name, label, tensor.mean(dim=0))
        except Exception:
            return None

    args_list = list(zip(df['feature_path'], df['slide_name'], df['label']))
    slide_features = defaultdict(lambda: defaultdict(list))

    print(f"  Loading {len(df):,} files with {MAX_WORKERS} threads...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(load_one, a): a for a in args_list}
        for future in tqdm(as_completed(futures), total=len(args_list), desc=f"  {model_name}"):
            result = future.result()
            if result is not None:
                slide_name, label, avg = result
                slide_features[slide_name][label].append(avg)

    print(f"  Loaded features for {len(slide_features):,} slides")

    # ── Aggregate + save ───────────────────────────────────────────────────────
    n_groups = len(ALL_GROUPS)
    for slide_name, group_data in tqdm(slide_features.items(), desc=f"  Aggregating {model_name}"):
        final_features = []
        for group in ALL_GROUPS:
            if group_data[group]:
                avg_features = torch.stack(group_data[group]).mean(dim=0, keepdim=True)
            else:
                avg_features = torch.zeros(1, feature_dim)
            final_features.append(avg_features)

        final_tensor = torch.cat(final_features, dim=0)   # (n_groups, feature_dim)
        torch.save(final_tensor, os.path.join(output_dir, f"{slide_name}.pt"))

    print(f"  ✓ Saved {len(slide_features):,} slide tensors ({n_groups} × {feature_dim}) → {output_dir}")


def main():
    print("📂 Loading CSV...")
    df = pd.read_csv(CSV_FILE)
    print(f"  Total rows     : {len(df):,}")

    df['patch_name_no_ext'] = df['patch_name'].str.replace('.png', '', regex=False)
    # PAIP: slide name is first 4 underscore-parts (e.g. TCGA-3L-AA1B_nonMSIH)
    df['slide_name'] = df['patch_name'].str.split('_').str[:4].str.join('_')

    print(f"  Unique slides  : {df['slide_name'].nunique():,}")
    print(f"  Labels found   : {sorted(df['label'].unique())}")
    print(f"\n  Active models  : {ACTIVE_MODELS}")

    for model_name in ACTIVE_MODELS:
        if model_name not in MODEL_CONFIG:
            print(f"\n[WARN] '{model_name}' not in MODEL_CONFIG — skipping.")
            continue
        run_for_model(model_name, MODEL_CONFIG[model_name], df)

    print(f"\n{'═'*60}")
    print("✅ ALL DONE")
    print(f"{'═'*60}")


if __name__ == "__main__":
    main()

📂 Loading CSV...
  Total rows     : 267,555
  Unique slides  : 73
  Labels found   : ['ADE', 'ADI', 'BACK', 'CAR', 'DEB', 'LYA', 'LYM', 'MUC', 'MUS', 'NORM', 'PDC', 'PLC', 'SIG', 'STR', 'TIL']

  Active models  : ['ConchV1', 'Conch1_5', 'H-Optimus-1', 'UNI2', 'Virchow2']

════════════════════════════════════════════════════════════
  MODEL: ConchV1
════════════════════════════════════════════════════════════
  Found : 267,044  |  Missing: 511
  Loading 267,044 files with 12 threads...


  ConchV1: 100%|██████████| 267044/267044 [03:34<00:00, 1245.21it/s]


  Loaded features for 73 slides


  Aggregating ConchV1: 100%|██████████| 73/73 [00:00<00:00, 96.32it/s] 


  ✓ Saved 73 slide tensors (15 × 512) → D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_based_aggregation_15_classes\ConchV1

════════════════════════════════════════════════════════════
  MODEL: Conch1_5
════════════════════════════════════════════════════════════
  Found : 267,044  |  Missing: 511
  Loading 267,044 files with 12 threads...


...

  ✓ Saved 73 slide tensors (15 × 1536) → D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_based_aggregation_15_classes\UNI2

════════════════════════════════════════════════════════════
  MODEL: Virchow2
════════════════════════════════════════════════════════════
  Found : 267,044  |  Missing: 511
  Loading 267,044 files with 12 threads...


  Virchow2: 100%|██████████| 267044/267044 [05:32<00:00, 802.01it/s] 


  Loaded features for 73 slides


  Aggregating Virchow2: 100%|██████████| 73/73 [00:01<00:00, 71.96it/s]


  ✓ Saved 73 slide tensors (15 × 2560) → D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_based_aggregation_15_classes\Virchow2

════════════════════════════════════════════════════════════
✅ ALL DONE
════════════════════════════════════════════════════════════


## Aggregation for PAIP data (Conch)

In [1]:
import pandas as pd
import torch
import os

# Paths to your CSV and feature vector directory
csv_file = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_aggregation\caption_generation\Results\classification_results_paip.csv"  # Your input CSV file
features_dir = r"D:\Aamir Gulzar\KSA_project2\paip_data\Features\Conch_v1_fivecrop_Updated" # Your feature directory
output_dir = r"D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_based_aggregation\Conch_v1"  # Adjust this path to save .pt files

# List of all groups (assuming you have 14 groups)
all_groups = ['ADI', 'DEB', 'LYM', 'PLC', 'LYA', 'MUC', 'MUS', 'NORM', 'ADE', 'STR', 'MES', 'CAR', 'PDC', 'SIG']

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(csv_file)

# Output the number of rows being used
print(f"Number of rows in the CSV file: {len(df)}")

# Initialize a dictionary to store slide-level averaged features
slide_features = {}

# Extract the slide names dynamically
slide_names_to_process = set(df['patch_name'].apply(lambda x: "_".join(x.split('_')[:2])))

# Filter the rows that correspond to the slides
filtered_df = df[df['patch_name'].apply(lambda x: "_".join(x.split('_')[:2]) in slide_names_to_process)]

# Output the number of rows being processed
print(f"Number of rows being processed: {len(filtered_df)}")

# Track the number of rows per slide
slide_row_counts = {}

# Process each row in the filtered DataFrame
for idx, row in filtered_df.iterrows():
    patch_name = row['patch_name']
    label = row['label']
    
    # Extract the slide name including MSIH or nonMSIH part from the patch name
    slide_name_parts = patch_name.split('_')[:4]  # Get the first two parts for the slide name (TCGA-3L-AA1B_nonMSIH)
    slide_name = "_".join(slide_name_parts)  # Join to form the full slide name with MSIH/nonMSIH
    
    # Remove the ".png" extension from the patch name (since it's not in the feature file name)
    patch_name_no_extension = patch_name.replace('.png', '')  # Remove '.png' from patch name
    
    # Construct the correct path to the feature vector .pt file
    # The feature file path should be like:
    # D:\Aamir Gulzar\KSA_project2\dataset\Features\Conch_v1_5_fivecrop_Updated\<slide_name>\<patch_name_no_extension>.pt
    feature_file = os.path.join(features_dir, slide_name, f"{patch_name_no_extension}.pt")
    
    if os.path.exists(feature_file):
        # Load the feature vector from the .pt file (assumes a 5x512 tensor)
        try:
            feature_tensor = torch.load(feature_file)
        except Exception as e:
            print(f"Error loading {feature_file}: {e}")
            continue
        
        # Average the 5x512 feature vector to get a 1x512 vector (by averaging the 5 rows)
        averaged_feature = torch.mean(feature_tensor, dim=0, keepdim=True)  # Shape: 1x512
        
        # Check if the slide_name already exists in the dictionary
        if slide_name not in slide_features:
            slide_features[slide_name] = {group: [] for group in all_groups}  # Initialize list for each group
        
        # Add the averaged feature to the appropriate group
        slide_features[slide_name][label].append(averaged_feature)
        
        # Track the number of rows for this slide_name
        if slide_name not in slide_row_counts:
            slide_row_counts[slide_name] = 0
        slide_row_counts[slide_name] += 1
    
    else:
        print(f"Feature file {feature_file} not found for {patch_name}, skipping...")

# Now, calculate the averaged features for all slides
for slide_name, group_data in slide_features.items():
    # Print the number of rows processed for this slide
    print(f"\nProcessing slide: {slide_name}, with {slide_row_counts[slide_name]} rows")
    
    final_features = []
    
    # For each group, if there are no features, append a zero vector; otherwise, average the 1x512 vectors
    for group in all_groups:
        if group_data[group]:  # If there are features for this group
            features = torch.cat(group_data[group], dim=0)  # Concatenate all 1x512 vectors for this group
            avg_features = torch.mean(features, dim=0, keepdim=True)  # Average the features for this group
        else:
            avg_features = torch.zeros(1, 512)  # Fill with zero vector if no features for this group (1x512 vector)
        
        final_features.append(avg_features)  # Append the averaged features for this group
    
    # Convert the final features list to a tensor (14x512 tensor)
    final_features_tensor = torch.cat(final_features, dim=0)  # Stack all group features into a tensor (14x512)
    
    # Save the averaged features tensor as a .pt file using just the slide_name
    output_file = os.path.join(output_dir, f"{slide_name}.pt")
    torch.save(final_features_tensor, output_file)
    print(f"Saved averaged features for {slide_name} to {output_file}")

print(f"\nAveraged features for the first n rows have been saved in '{output_dir}'")

Number of rows in the CSV file: 271322
Number of rows being processed: 271322

Processing slide: training_data_01_MSIH, with 3196 rows
Saved averaged features for training_data_01_MSIH to D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_based_aggregation\Conch_v1\training_data_01_MSIH.pt

Processing slide: training_data_02_nonMSIH, with 4412 rows
Saved averaged features for training_data_02_nonMSIH to D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_based_aggregation\Conch_v1\training_data_02_nonMSIH.pt

Processing slide: training_data_03_nonMSIH, with 1955 rows
Saved averaged features for training_data_03_nonMSIH to D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_based_aggregation\Conch_v1\training_data_03_nonMSIH.pt

Processing slide: training_data_04_nonMSIH, with 3198 rows
Saved averaged features for training_data_04_nonMSIH to D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_based_aggregation\Conch_v1\training_data_0

## Aggregation for PAIP data (UNI2)

In [2]:
import pandas as pd
import torch
import os

# Paths to your CSV and feature vector directory
csv_file = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Other_Aggregations\caption_generation\Results\classification_results_paip.csv"  # Your input CSV file
features_dir = r"D:\Aamir Gulzar\KSA_project2\paip_data\Features\Uni_2h_fivecrop_Updated" # Your feature directory
output_dir = r"D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_Based_Clustering_FiveCrop\uni2_CC_fivecrop"  # Adjust this path to save .pt files

# List of all groups (assuming you have 14 groups)
all_groups = ['ADI', 'DEB', 'LYM', 'PLC', 'LYA', 'MUC', 'MUS', 'NORM', 'ADE', 'STR', 'MES', 'CAR', 'PDC', 'SIG']

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(csv_file)

# Output the number of rows being used
print(f"Number of rows in the CSV file: {len(df)}")

# Initialize a dictionary to store slide-level averaged features
slide_features = {}

# Extract the slide names dynamically
slide_names_to_process = set(df['patch_name'].apply(lambda x: "_".join(x.split('_')[:2])))

# Filter the rows that correspond to the slides
filtered_df = df[df['patch_name'].apply(lambda x: "_".join(x.split('_')[:2]) in slide_names_to_process)]

# Output the number of rows being processed
print(f"Number of rows being processed: {len(filtered_df)}")

# Track the number of rows per slide
slide_row_counts = {}

# Process each row in the filtered DataFrame
for idx, row in filtered_df.iterrows():
    patch_name = row['patch_name']
    label = row['label']
    
    # Extract the slide name including MSIH or nonMSIH part from the patch name
    slide_name_parts = patch_name.split('_')[:4]  # Get the first two parts for the slide name (TCGA-3L-AA1B_nonMSIH)
    slide_name = "_".join(slide_name_parts)  # Join to form the full slide name with MSIH/nonMSIH
    
    # Remove the ".png" extension from the patch name (since it's not in the feature file name)
    patch_name_no_extension = patch_name.replace('.png', '')  # Remove '.png' from patch name
    
    # Construct the correct path to the feature vector .pt file
    # The feature file path should be like:
    # D:\Aamir Gulzar\KSA_project2\dataset\Features\Conch_v1_5_fivecrop_Updated\<slide_name>\<patch_name_no_extension>.pt
    feature_file = os.path.join(features_dir, slide_name, f"{patch_name_no_extension}.pt")
    
    if os.path.exists(feature_file):
        # Load the feature vector from the .pt file (assumes a 5x768 tensor)
        try:
            feature_tensor = torch.load(feature_file)
        except Exception as e:
            print(f"Error loading {feature_file}: {e}")
            continue
        
        # Average the 5x768 feature vector to get a 1x768 vector (by averaging the 5 rows)
        averaged_feature = torch.mean(feature_tensor, dim=0, keepdim=True)  # Shape: 1x768
        
        # Check if the slide_name already exists in the dictionary
        if slide_name not in slide_features:
            slide_features[slide_name] = {group: [] for group in all_groups}  # Initialize list for each group
        
        # Add the averaged feature to the appropriate group
        slide_features[slide_name][label].append(averaged_feature)
        
        # Track the number of rows for this slide_name
        if slide_name not in slide_row_counts:
            slide_row_counts[slide_name] = 0
        slide_row_counts[slide_name] += 1
    
    else:
        print(f"Feature file {feature_file} not found for {patch_name}, skipping...")

# Now, calculate the averaged features for all slides
for slide_name, group_data in slide_features.items():
    # Print the number of rows processed for this slide
    print(f"\nProcessing slide: {slide_name}, with {slide_row_counts[slide_name]} rows")
    
    final_features = []
    
    # For each group, if there are no features, append a zero vector; otherwise, average the 1x768 vectors
    for group in all_groups:
        if group_data[group]:  # If there are features for this group
            features = torch.cat(group_data[group], dim=0)  # Concatenate all 1x768 vectors for this group
            avg_features = torch.mean(features, dim=0, keepdim=True)  # Average the features for this group
        else:
            avg_features = torch.zeros(1, 1536)  # Fill with zero vector if no features for this group (1x768 vector)
        
        final_features.append(avg_features)  # Append the averaged features for this group
    
    # Convert the final features list to a tensor (14x768 tensor)
    final_features_tensor = torch.cat(final_features, dim=0)  # Stack all group features into a tensor (14x768)
    
    # Save the averaged features tensor as a .pt file using just the slide_name
    output_file = os.path.join(output_dir, f"{slide_name}.pt")
    torch.save(final_features_tensor, output_file)
    print(f"Saved averaged features for {slide_name} to {output_file}")

print(f"\nAveraged features for the first n rows have been saved in '{output_dir}'")

Number of rows in the CSV file: 271322
Number of rows being processed: 271322

Processing slide: training_data_01_MSIH, with 3196 rows
Saved averaged features for training_data_01_MSIH to D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_Based_Clustering_FiveCrop\uni2_CC_fivecrop\training_data_01_MSIH.pt

Processing slide: training_data_02_nonMSIH, with 4412 rows
Saved averaged features for training_data_02_nonMSIH to D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_Based_Clustering_FiveCrop\uni2_CC_fivecrop\training_data_02_nonMSIH.pt

Processing slide: training_data_03_nonMSIH, with 1955 rows
Saved averaged features for training_data_03_nonMSIH to D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_Based_Clustering_FiveCrop\uni2_CC_fivecrop\training_data_03_nonMSIH.pt

Processing slide: training_data_04_nonMSIH, with 3198 rows
Saved averaged features for training_data_04_nonMSIH to D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Ca

## Aggregation for PAIP data (Hoptimus1)

In [3]:
import pandas as pd
import torch
import os

# Paths to your CSV and feature vector directory
csv_file = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Other_Aggregations\caption_generation\Results\classification_results_paip.csv"  # Your input CSV file
features_dir = r"D:\Aamir Gulzar\KSA_project2\paip_data\Features\H_Optimus_1_fivecrop_Updated" # Your feature directory
output_dir = r"D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_Based_Clustering_FiveCrop\hoptimus_CC_fivecrop"  # Adjust this path to save .pt files

# List of all groups (assuming you have 14 groups)
all_groups = ['ADI', 'DEB', 'LYM', 'PLC', 'LYA', 'MUC', 'MUS', 'NORM', 'ADE', 'STR', 'MES', 'CAR', 'PDC', 'SIG']

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(csv_file)

# Output the number of rows being used
print(f"Number of rows in the CSV file: {len(df)}")

# Initialize a dictionary to store slide-level averaged features
slide_features = {}

# Extract the slide names dynamically
slide_names_to_process = set(df['patch_name'].apply(lambda x: "_".join(x.split('_')[:2])))

# Filter the rows that correspond to the slides
filtered_df = df[df['patch_name'].apply(lambda x: "_".join(x.split('_')[:2]) in slide_names_to_process)]

# Output the number of rows being processed
print(f"Number of rows being processed: {len(filtered_df)}")

# Track the number of rows per slide
slide_row_counts = {}

# Process each row in the filtered DataFrame
for idx, row in filtered_df.iterrows():
    patch_name = row['patch_name']
    label = row['label']
    
    # Extract the slide name including MSIH or nonMSIH part from the patch name
    slide_name_parts = patch_name.split('_')[:4]  # Get the first two parts for the slide name (TCGA-3L-AA1B_nonMSIH)
    slide_name = "_".join(slide_name_parts)  # Join to form the full slide name with MSIH/nonMSIH
    
    # Remove the ".png" extension from the patch name (since it's not in the feature file name)
    patch_name_no_extension = patch_name.replace('.png', '')  # Remove '.png' from patch name
    
    # Construct the correct path to the feature vector .pt file
    # The feature file path should be like:
    # D:\Aamir Gulzar\KSA_project2\dataset\Features\Conch_v1_5_fivecrop_Updated\<slide_name>\<patch_name_no_extension>.pt
    feature_file = os.path.join(features_dir, slide_name, f"{patch_name_no_extension}.pt")
    
    if os.path.exists(feature_file):
        # Load the feature vector from the .pt file (assumes a 5x768 tensor)
        try:
            feature_tensor = torch.load(feature_file)
        except Exception as e:
            print(f"Error loading {feature_file}: {e}")
            continue
        
        # Average the 5x768 feature vector to get a 1x768 vector (by averaging the 5 rows)
        averaged_feature = torch.mean(feature_tensor, dim=0, keepdim=True)  # Shape: 1x768
        
        # Check if the slide_name already exists in the dictionary
        if slide_name not in slide_features:
            slide_features[slide_name] = {group: [] for group in all_groups}  # Initialize list for each group
        
        # Add the averaged feature to the appropriate group
        slide_features[slide_name][label].append(averaged_feature)
        
        # Track the number of rows for this slide_name
        if slide_name not in slide_row_counts:
            slide_row_counts[slide_name] = 0
        slide_row_counts[slide_name] += 1
    
    else:
        print(f"Feature file {feature_file} not found for {patch_name}, skipping...")

# Now, calculate the averaged features for all slides
for slide_name, group_data in slide_features.items():
    # Print the number of rows processed for this slide
    print(f"\nProcessing slide: {slide_name}, with {slide_row_counts[slide_name]} rows")
    
    final_features = []
    
    # For each group, if there are no features, append a zero vector; otherwise, average the 1x768 vectors
    for group in all_groups:
        if group_data[group]:  # If there are features for this group
            features = torch.cat(group_data[group], dim=0)  # Concatenate all 1x768 vectors for this group
            avg_features = torch.mean(features, dim=0, keepdim=True)  # Average the features for this group
        else:
            avg_features = torch.zeros(1, 1536)  # Fill with zero vector if no features for this group (1x768 vector)
        
        final_features.append(avg_features)  # Append the averaged features for this group
    
    # Convert the final features list to a tensor (14x768 tensor)
    final_features_tensor = torch.cat(final_features, dim=0)  # Stack all group features into a tensor (14x768)
    
    # Save the averaged features tensor as a .pt file using just the slide_name
    output_file = os.path.join(output_dir, f"{slide_name}.pt")
    torch.save(final_features_tensor, output_file)
    print(f"Saved averaged features for {slide_name} to {output_file}")

print(f"\nAveraged features for the first n rows have been saved in '{output_dir}'")

Number of rows in the CSV file: 271322
Number of rows being processed: 271322

Processing slide: training_data_01_MSIH, with 3196 rows
Saved averaged features for training_data_01_MSIH to D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_Based_Clustering_FiveCrop\hoptimus_CC_fivecrop\training_data_01_MSIH.pt

Processing slide: training_data_02_nonMSIH, with 4412 rows
Saved averaged features for training_data_02_nonMSIH to D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_Based_Clustering_FiveCrop\hoptimus_CC_fivecrop\training_data_02_nonMSIH.pt

Processing slide: training_data_03_nonMSIH, with 1955 rows
Saved averaged features for training_data_03_nonMSIH to D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_Based_Clustering_FiveCrop\hoptimus_CC_fivecrop\training_data_03_nonMSIH.pt

Processing slide: training_data_04_nonMSIH, with 3198 rows
Saved averaged features for training_data_04_nonMSIH to D:\Aamir Gulzar\KSA_project2\paip_data\slide_ag

## Analyze patch .pt file

In [ ]:
import torch

# Path to your .pt file
file_path = r"D:\Aamir Gulzar\KSA_project2\paip_data\Features\Conch_v1_5_fivecrop_Updated\training_data_08_nonMSIH\training_data_08_nonMSIH_x26757_y12544_patch00000.pt"  # change this
# file_path = r"D:\Aamir Gulzar\KSA_project2\paip_data\Features\Conch_v1_5_fivecrop_Updated\training_data_07_nonMSIH\training_data_07_nonMSIH_x1616_y22769_patch00000.pt"
obj = torch.load(file_path, map_location="cpu")

# Case 1: if it's a plain tensor
if isinstance(obj, torch.Tensor):
    print("📦 This file contains a single tensor.\n")
    print(f"Shape: {tuple(obj.shape)}")
    print(f"Sample values: {obj.flatten()[:10].tolist()}")

# Case 2: if it's a dictionary (checkpoint or state_dict)
elif isinstance(obj, dict):
    print("📦 This file contains a dictionary.\n")
    # Try to detect where weights are stored
    if 'model_state_dict' in obj:
        state_dict = obj['model_state_dict']
    elif 'state_dict' in obj:
        state_dict = obj['state_dict']
    else:
        state_dict = obj

    if isinstance(state_dict, dict):
        print(f"Total tensors: {len(state_dict)}\n")
        for i, (name, tensor) in enumerate(state_dict.items()):
            print(f"{name:40s} shape={tuple(tensor.shape)}")
            print(f"  sample values: {tensor.flatten()[:5].tolist()}\n")
            if i >= 9:  # limit output
                print("... (showing first 10 tensors only)")
                break
    else:
        print("Dictionary does not contain tensor values directly.")

# Case 3: if it’s a full model
elif hasattr(obj, "state_dict"):
    print("📦 This file contains a full model.\n")
    state_dict = obj.state_dict()
    print(f"Total tensors: {len(state_dict)}\n")
    for i, (name, tensor) in enumerate(state_dict.items()):
        print(f"{name:40s} shape={tuple(tensor.shape)}")
        print(f"  sample values: {tensor.flatten()[:5].tolist()}\n")
        if i >= 9:
            print("... (showing first 10 tensors only)")
            break

else:
    print(f"❌ Unknown object type: {type(obj)}")


📦 This file contains a single tensor.

Shape: (5, 768)
Sample values: [-1.0991148948669434, 0.17387989163398743, -3.683518409729004, 1.163424015045166, -1.6222351789474487, 0.6189315319061279, 2.1375374794006348, -0.02675045281648636, 0.5801745057106018, -0.25191885232925415]


## SurGen data aggregation